In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import joblib

TRAIN_PATH = '/kaggle/input/datasets/antonoof/traindata-media/train.csv'
MODEL_DIR = '/kaggle/working/models'
os.makedirs(MODEL_DIR, exist_ok=True)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Удаляем спецсимволы, оставляем буквы, цифры и пробелы
    text = re.sub(r'[^a-zа-яё0-9\s]', ' ', text)
    # Удаляем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv(TRAIN_PATH)
df['QueryText'] = df['QueryText'].apply(clean_text)

df_train_type = df.copy()
df_train_content = df[df['TypeQuery'] == 1].copy()

tfidf_type = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_type = tfidf_type.fit_transform(df_train_type['QueryText'])
y_type = df_train_type['TypeQuery']

# Используем LogisticRegression для калиброванных вероятностей или SGD для скорости
model_type = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
model_type.fit(X_type, y_type)

with open(os.path.join(MODEL_DIR, 'tfidf_type.pkl'), 'wb') as f:
    pickle.dump(tfidf_type, f)
with open(os.path.join(MODEL_DIR, 'model_type.pkl'), 'wb') as f:
    pickle.dump(model_type, f)


if not df_train_content.empty:
    le_content = LabelEncoder()
    y_content_enc = le_content.fit_transform(df_train_content['ContentType'])
    
    tfidf_content = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), sublinear_tf=True)
    X_content = tfidf_content.fit_transform(df_train_content['QueryText'])
    
    model_content = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
    model_content.fit(X_content, y_content_enc)
    
    with open(os.path.join(MODEL_DIR, 'tfidf_content.pkl'), 'wb') as f:
        pickle.dump(tfidf_content, f)
    with open(os.path.join(MODEL_DIR, 'model_content.pkl'), 'wb') as f:
        pickle.dump(model_content, f)
    with open(os.path.join(MODEL_DIR, 'le_content.pkl'), 'wb') as f:
        pickle.dump(le_content, f)
else:
    print("Warning: No positive samples for ContentType training.")


stop_words_search = {
    'смотреть', 'онлайн', 'бесплатно', 'скачать', 'торрент', 'фильм', 'сериал', 
    'мультик', 'мультфильм', 'аниме', 'дорама', 'читать', 'книга', 'слушать',
    'трейлер', 'отрывок', 'фрагмент', 'клип', 'песня', 'текст', 'перевод',
    'озвучка', 'дубляж', 'сезон', 'серия', 'выпуск', 'год', 'дата', 'качество',
    'hd', 'full', 'hdrip', 'ts', 'camrip', 'webrip', 'avi', 'mp4', 'mkv',
    'в хорошем', 'качестве', 'на русском', 'english', 'subtitles', 'sub',
    'все серии', 'последняя серия', 'новый', 'новинка', '2023', '2024', '2025', '2026',
    'купить', 'цена', 'заказать', 'магазин', 'интернет', 'сайт', 'страница',
    'фото', 'картинка', 'изображение', 'видео', 'аудио', 'мп3', 'mp3',
    'реферат', 'доклад', 'сочинение', 'ответы', 'гдз', 'решебник',
    'своими руками', 'мастер класс', 'урок', 'курс', 'обучение',
    'как', 'что', 'где', 'когда', 'почему', 'зачем', 'кто', 'который',
    'и', 'в', 'на', 'с', 'по', 'к', 'у', 'о', 'из', 'за', 'под', 'над', 'для',
    'не', 'ни', 'бы', 'ли', 'же', 'быть', 'был', 'была', 'было', 'были',
    'этот', 'эта', 'это', 'эти', 'тот', 'та', 'то', 'те',
    'мой', 'твой', 'свой', 'наш', 'ваш', 'их',
    'сам', 'сама', 'само', 'сами',
    'так', 'также', 'тоже', 'уже', 'еще', 'более', 'менее', 'очень',
    'можно', 'нужно', 'надо', 'следует', 'стоит',
    'смотреть онлайн', 'смотреть бесплатно', 'скачать торрент', 'скачать бесплатно',
    'фильм смотреть', 'сериал смотреть', 'аниме смотреть',
    'онлайн бесплатно', 'бесплатно онлайн',
    'русский перевод', 'оригинал', 'озвучка',
    '1 сезон', '2 сезон', '3 сезон', '4 сезон', '5 сезон',
    '1 серия', '2 серия', '3 серия', '4 серия', '5 серия',
    'полный фильм', 'полный сериал',
    'лучшие', 'топ', 'рейтинг',
    'новости', 'статья', 'обзор', 'рецензия',
    'купить билет', 'расписание', 'афиша',
    'скачать игру', 'игра онлайн', 'играть бесплатно',
    'рецепт', 'приготовление', 'блюдо',
    'погода', 'прогноз', 'температура',
    'курс валют', 'доллар', 'евро', 'рубль',
    'новости сегодня', 'последние новости',
    'рецепты', 'готовим дома',
    'сонник', 'гороскоп', 'знаки зодиака',
    'приметы', 'гадание',
    'здоровье', 'симптомы', 'лечение', 'лекарства',
    'юридическая помощь', 'консультация', 'адвокат',
    'работа', 'вакансии', 'резюме', 'собеседование',
    'недвижимость', 'квартира', 'дом', 'аренда', 'продажа',
    'авто', 'машина', 'автомобиль', 'купить авто', 'продажа авто',
    'туризм', 'отель', 'гостиница', 'путевка', 'виза',
    'образование', 'университет', 'школа', 'курсы', 'языки',
    'техника', 'смартфон', 'ноутбук', 'компьютер', 'телефон',
    'одежда', 'обувь', 'мода', 'стиль',
    'красота', 'косметика', 'парфюмерия', 'уход',
    'спорт', 'футбол', 'хоккей', 'баскетбол', 'теннис',
    'музыка', 'концерт', 'альбом', 'исполнитель',
    'игры', 'playstation', 'xbox', 'pc', 'steam',
    'программы', 'софт', 'windows', 'android', 'ios',
    'наука', 'космос', 'природа', 'животные', 'растения',
    'история', 'биография', 'личность',
    'психология', 'отношения', 'семья', 'дети',
    'финансы', 'банк', 'кредит', 'ипотека', 'страховка',
    'бизнес', 'маркетинг', 'реклама', 'продажи',
    'строительство', 'ремонт', 'дизайн', 'интерьер',
    'сад', 'огород', 'дача', 'цветы',
    'рыбалка', 'охота', 'туризм', 'походы',
    'юмор', 'анекдоты', 'мемы', 'приколы',
    'гороскоп', 'астрология', 'эзотерика',
    'религия', 'православие', 'ислам', 'буддизм',
    'политика', 'выборы', 'партии', 'лидеры',
    'армия', 'военная служба', 'оружие',
    'транспорт', 'общественный транспорт', 'метро', 'автобус',
    'связь', 'интернет', 'мобильная связь', 'тарифы',
    'ЖКХ', 'коммунальные услуги', 'квартплата',
    'пенсия', 'пособия', 'льготы', 'соцзащита',
    'МФЦ', 'госуслуги', 'документы', 'паспорт', 'visa',
    'суд', 'полиция', 'гибдд', 'налоги', 'штрафы',
    'ЧС', 'пожар', 'МЧС', 'безопасность',
    'экология', 'загрязнение', 'переработка',
    'волонтерство', 'благотворительность', 'помощь',
    'хобби', 'рукоделие', 'коллекционирование',
    'танцы', 'театр', 'кино', 'литература', 'живопись',
    'фотография', 'видеосъемка', 'монтаж',
    'кулинария', 'выпечка', 'напитки', 'коктейли',
    'диеты', 'похудение', 'фитнес', 'йога',
    'беременность', 'роды', 'новорожденные',
    'воспитание', 'развитие', 'школьники', 'студенты',
    'профессии', 'карьера', 'саморазвитие',
    'инвестиции', 'биржа', 'криптовалюта', 'трейдинг',
    'недвижимость за рубежом', 'иммиграция', 'ПМЖ',
    'брак', 'свадьба', 'развод', 'алименты',
    'наследство', 'дарственная', 'купля-продажа',
    'арбитраж', 'банкротство', 'долги',
    'патенты', 'авторское право', 'лицензии',
    'сертификация', 'стандарты', 'ГОСТ',
    'логистика', 'доставка', 'таможня', 'импорт', 'экспорт',
    'производство', 'сырье', 'оборудование',
    'сельское хозяйство', 'фермерство', 'животноводство',
    'ветеринария', 'зоомагазины', 'груминг',
    'психиатрия', 'наркологи', 'реабилитация',
    'стоматология', 'офтальмология', 'кардиология',
    'дерматология', 'эндокринология', 'неврология',
    'педиатрия', 'гинекология', 'урология',
    'хирургия', 'терапия', 'диагностика', 'анализы',
    'аптеки', 'лекарственные препараты', 'БАДы',
    'медицинские услуги', 'страхование жизни',
    'корпоративное обучение', 'тренинги', 'семинары',
    'конференции', 'выставки', 'форумы',
    'нетворкинг', 'коммуникации', 'переговоры',
    'управление персоналом', 'HR', 'рекрутинг',
    'бухгалтерия', 'аудит', 'налогообложение',
    'юридические услуги', 'нотариус', 'адвокатура',
    'оценка недвижимости', 'риэлторы', 'ипотека',
    'ландшафтный дизайн', 'благоустройство',
    'клининг', 'уборка', 'химчистка',
    'ритуальные услуги', 'похороны', 'памятники',
    'агентства недвижимости', 'застройщики',
    'аренда оборудования', 'прокат', 'лизинг',
    'ремонт техники', 'сервисные центры',
    'установка ПО', 'настройка компьютеров',
    'создание сайтов', 'веб-дизайн', 'SEO',
    'SMM', 'таргетированная реклама', 'контекстная реклама',
    'email-рассылки', 'CRM-системы',
    'облачные технологии', 'Big Data', 'AI', 'Machine Learning',
    'блокчейн', 'смарт-контракты', 'NFT',
    'VR', 'AR', 'IoT', 'робототехника',
    'биоинженерия', 'генетика', 'нанотехнологии',
    'возобновляемая энергетика', 'солнечные панели',
    'электромобили', 'автопилот', 'космические полеты',
    '3D-печать', 'дроны', 'квадрокоптеры',
    'умный дом', 'системы безопасности',
    'видеонаблюдение', 'домофоны', 'сигнализации',
    'пожарная безопасность', 'охрана труда',
    'гражданская оборона', 'действия при ЧС',
    'первая помощь', 'спасательные работы',
    'поисково-спасательные службы',
    'волонтерские организации', 'НКО', 'фонды',
    'гранты', 'субсидии', 'социальные программы',
    'государственные услуги', 'электронное правительство',
    'цифровизация', 'информационные технологии',
    'телекоммуникации', 'спутниковая связь',
    'радиосвязь', 'оптоволокно', '5G', '6G',
    'кибербезопасность', 'защита данных', 'шифрование',
    'хакинг', 'вирусы', 'антивирусы',
    'социальная инженерия', 'фишинг', 'мошенничество',
    'правоохранительные органы', 'следственный комитет',
    'прокуратура', 'верховный суд', 'конституционный суд',
    'международное право', 'Европейский суд',
    'ООН', 'НАТО', 'ЕС', 'АСЕАН', 'БРИКС',
    'геополитика', 'международные отношения',
    'дипломатия', 'посольства', 'консульства',
    'визовые центры', 'миграционная служба',
    'беженцы', 'вынужденные переселенцы',
    'гуманитарная помощь', 'миротворческие миссии',
    'военные конфликты', 'терроризм', 'экстремизм',
    'сепаратизм', 'национализм', 'шовинизм',
    'расизм', 'дискриминация', 'толерантность',
    'права человека', 'свобода слова', 'цензура',
    'демократия', 'тоталитаризм', 'авторитаризм',
    'монархия', 'республика', 'федерация',
    'парламент', 'президент', 'правительство',
    'министерства', 'ведомства', 'агентства',
    'муниципалитеты', 'местное самоуправление',
    'выборы президента', 'выборы в парламент',
    'референдум', 'плебисцит', 'инициатива',
    'политические партии', 'движения', 'лидеры мнений',
    'общественные организации', 'профсоюзы',
    'забастовки', 'митинги', 'демонстрации',
    'революции', 'перевороты', 'реформы',
    'приватизация', 'национализация', 'санкции',
    'эмбарго', 'блокада', 'бойкот',
    'торговые войны', 'валютные войны',
    'инфляция', 'дефляция', 'стагфляция',
    'рецессия', 'депрессия', 'кризис',
    'экономический рост', 'ВВП', 'ВНП',
    'безработица', 'занятость', 'трудовые ресурсы',
    'производительность труда', 'конкуренция',
    'монополия', 'олигополия', 'картель',
    'биржи', 'акции', 'облигации', 'фьючерсы',
    'опционы', 'хедж-фонды', 'венчурные фонды',
    'стартапы', 'инкубаторы', 'акселераторы',
    'IPO', 'SPAC', 'слияния и поглощения',
    'корпоративное управление', 'совет директоров',
    'акционеры', 'дивиденды', 'капитализация',
    'ликвидность', 'рентабельность', 'маржинальность',
    'себестоимость', 'прибыль', 'убыток',
    'баланс', 'отчет о прибылях и убытках',
    'cash flow', 'EBITDA', 'ROI', 'ROE',
    'KPI', 'OKR', 'SMART', 'SWOT',
    'бенчмаркинг', 'аутсорсинг', 'офшоры',
    'трансфертное ценообразование', 'налоговая оптимизация',
    'бухгалтерский учет', 'управленческий учет',
    'финансовый анализ', 'инвестиционный анализ',
    'риск-менеджмент', 'комплаенс', 'due diligence',
    'юридическая экспертиза', 'технический аудит',
    'энергоаудит', 'экологический аудит',
    'сертификация ISO', 'GMP', 'HACCP',
    'лицензирование', 'аккредитация', 'аттестация',
    'повышение квалификации', 'переподготовка',
    'дипломы', 'сертификаты', 'аттестаты',
    'ученые степени', 'звания', 'награды',
    'гранты на исследования', 'научные статьи',
    'патенты на изобретения', 'ноу-хау',
    'технологический трансфер', 'инновации',
    'R&D', 'исследования и разработки',
    'прототипирование', 'тестирование', 'валидация',
    'внедрение', 'масштабирование', 'коммерциализация',
    'маркетинговые исследования', 'фокус-группы',
    'опросы', 'анкетирование', 'статистика',
    'социология', 'демография', 'урбанистика',
    'картография', 'геодезия', 'геология',
    'метеорология', 'климатология', 'океанология',
    'биология', 'ботаника', 'зоология', 'микробиология',
    'вирусология', 'иммунология', 'генетика',
    'эволюция', 'экология', 'биогеоценоз',
    'заповедники', 'национальные парки', 'красная книга',
    'браконьерство', 'охранные зоны',
    'переработка отходов', 'вторсырье', 'утилизация',
    'чистая энергия', 'ветрогенераторы', 'ГЭС', 'АЭС',
    'термоядерный синтез', 'водородная энергетика',
    'аккумуляторы', 'батареи', 'зарядные устройства',
    'электросети', 'smart grid', 'энергосбережение',
    'теплоизоляция', 'энергоэффективные дома',
    'пассивные дома', 'зеленое строительство',
    'LEED', 'BREEAM', 'DGNB',
    'умные города', 'интеллектуальные транспортные системы',
    'беспилотный транспорт', 'каршеринг', 'райдинг',
    'логистика последней мили', 'дроны-курьеры',
    'роботы-курьеры', 'автоматизация складов',
    'WMS', 'TMS', 'ERP', 'CRM', 'SCM',
    'big data analytics', 'data mining', 'machine learning ops',
    'deep learning', 'neural networks', 'computer vision',
    'natural language processing', 'speech recognition',
    'generative AI', 'LLM', 'transformers', 'diffusion models',
    'GAN', 'VAE', 'reinforcement learning',
    'компьютерная графика', 'рендеринг', 'мокапы',
    'CGI', 'VFX', 'motion design', '3D modeling',
    'game dev', 'unity', 'unreal engine', 'godot',
    'mobile dev', 'iOS', 'android', 'flutter', 'react native',
    'web dev', 'frontend', 'backend', 'fullstack',
    'javascript', 'typescript', 'python', 'java', 'c++', 'c#',
    'go', 'rust', 'swift', 'kotlin', 'php', 'ruby',
    'sql', 'nosql', 'postgresql', 'mysql', 'mongodb', 'redis',
    'docker', 'kubernetes', 'ci/cd', 'devops', 'sre',
    'cloud computing', 'aws', 'azure', 'gcp', 'yandex cloud',
    'serverless', 'microservices', 'api', 'graphql', 'rest',
    'blockchain development', 'solidity', 'web3', 'defi', 'dao',
    'cybersecurity operations', 'pentesting', 'bug bounty',
    'soc', 'siem', 'ids/ips', 'firewalls', 'vpn',
    'identity management', 'mfa', 'sso', 'iam',
    'gdpr', 'ccpa', 'personal data protection',
    'digital ethics', 'ai ethics', 'bias in ai',
    'explainable ai', 'fairness', 'accountability',
    'transparency', 'privacy by design',
    'human-computer interaction', 'ux/ui design',
    'accessibility', 'inclusive design',
    'product management', 'agile', 'scrum', 'kanban',
    'lean startup', 'design thinking', 'customer development',
    'growth hacking', 'viral marketing', 'content marketing',
    'influencer marketing', 'affiliate marketing',
    'performance marketing', 'brand management',
    'pr', 'crisis management', 'reputation management',
    'event management', 'mice industry',
    'hospitality industry', 'tourism management',
    'restaurant business', 'food service', 'catering',
    'retail', 'e-commerce', 'marketplaces', 'omnichannel',
    'supply chain management', 'procurement', 'inventory management',
    'quality management', 'six sigma', 'lean manufacturing',
    'industry 4.0', 'digital transformation',
    'internet of things', 'industrial iot',
    'predictive maintenance', 'digital twins',
    'augmented reality in industry', 'virtual reality training',
    'remote work', 'hybrid work', 'digital nomads',
    'freelance', 'gig economy', 'platform economy',
    'sharing economy', 'subscription economy',
    'creator economy', 'personal branding',
    'lifelong learning', 'upskilling', 'reskilling',
    'mental health', 'wellbeing', 'work-life balance',
    'burnout prevention', 'stress management',
    'emotional intelligence', 'soft skills',
    'leadership', 'team building', 'conflict resolution',
    'negotiation skills', 'public speaking',
    'time management', 'productivity', 'goal setting',
    'mindfulness', 'meditation', 'yoga for beginners',
    'healthy lifestyle', 'nutrition', 'sleep hygiene',
    'physical activity', 'sports nutrition',
    'recovery', 'massage', 'physiotherapy',
    'alternative medicine', 'homeopathy', 'acupuncture',
    'traditional medicine', 'integrative medicine',
    'preventive medicine', 'check-ups', 'screening',
    'vaccination', 'epidemiology', 'public health',
    'global health', 'health equity', 'access to care',
    'medical tourism', 'telemedicine', 'digital health',
    'wearables', 'health apps', 'fitness trackers',
    'biohacking', 'longevity', 'anti-aging',
    'regenerative medicine', 'stem cells', 'gene therapy',
    'personalized medicine', 'precision oncology',
    'neurotechnology', 'brain-computer interfaces',
    'prosthetics', 'exoskeletons', 'assistive technologies',
    'space medicine', 'aviation medicine',
    'sports medicine', 'orthopedics', 'traumatology',
    'radiology', 'mri', 'ct scan', 'ultrasound',
    'laboratory diagnostics', 'pathology', 'histology',
    'pharmacology', 'clinical trials', 'drug development',
    'medical devices', 'implants', 'biomaterials',
    'tissue engineering', 'organ printing',
    'synthetic biology', 'bioinformatics',
    'computational biology', 'systems biology',
    'structural biology', 'molecular biology',
    'cell biology', 'developmental biology',
    'physiology', 'anatomy', 'embryology',
    'parasitology', 'mycology', 'algology',
    'lichenology', 'bryology', 'pteridology',
    'pomology', 'ampelography', 'oenology',
    'brewing', 'distilling', 'fermentation',
    'food science', 'food technology', 'food safety',
    'packaging', 'labeling', 'traceability',
    'organic food', 'non-gmo', 'gluten-free',
    'vegan', 'vegetarian', 'pescatarian',
    'ketogenic diet', 'paleo diet', 'mediterranean diet',
    'intermittent fasting', 'detox', 'cleansing',
    'supplements', 'vitamins', 'minerals',
    'probiotics', 'prebiotics', 'enzymes',
    'adaptogens', 'nootropics', 'herbal remedies',
    'essential oils', 'aromatherapy', 'phytotherapy',
    'ayurveda', 'tcm', 'unani', 'siddha',
    'naturopathy', 'osteopathy', 'chiropractic',
    'reflexology', 'reiki', 'qigong', 'tai chi',
    'pilates', 'stretching', 'calisthenics',
    'crossfit', 'powerlifting', 'bodybuilding',
    'strongman', 'weightlifting', 'athletics',
    'swimming', 'diving', 'water polo',
    'synchronized swimming', 'rowing', 'canoeing',
    'sailing', 'windsurfing', 'kitesurfing',
    'surfing', 'skateboarding', 'snowboarding',
    'skiing', 'figure skating', 'speed skating',
    'hockey', 'bandy', 'lacrosse', 'rugby',
    'american football', 'baseball', 'softball',
    'cricket', 'volleyball', 'basketball',
    'handball', 'futsal', 'beach soccer',
    'table tennis', 'badminton', 'squash',
    'tennis', 'golf', 'bowling', 'billiards',
    'darts', 'archery', 'shooting', 'fencing',
    'judo', 'karate', 'taekwondo', 'aikido',
    'kung fu', 'wushu', 'sambo', 'wrestling',
    'boxing', 'kickboxing', 'muay thai',
    'mma', 'bjj', 'capoeira', 'parkour',
    'rock climbing', 'mountaineering', 'hiking',
    'cycling', 'mtb', 'road cycling', 'triathlon',
    'marathon', 'ultra-marathon', 'trail running',
    'orienteering', 'rogaining', 'geocaching',
    'fishing', 'hunting', 'bird watching',
    'photography safaris', 'wildlife photography',
    'landscape photography', 'street photography',
    'portrait photography', 'macro photography',
    'astrophotography', 'underwater photography',
    'aerial photography', 'drone photography',
    'photo editing', 'lightroom', 'photoshop',
    'video editing', 'premiere pro', 'final cut',
    'davinci resolve', 'after effects',
    'sound design', 'audio editing', 'mixing',
    'mastering', 'podcasting', 'radio broadcasting',
    'tv production', 'film production',
    'scriptwriting', 'directing', 'acting',
    'casting', 'production design', 'costume design',
    'makeup artistry', 'hair styling',
    'modeling', 'fashion design', 'textile design',
    'jewelry design', 'interior design',
    'graphic design', 'illustration', 'comics',
    'animation', 'stop motion', 'claymation',
    'puppetry', 'magic tricks', 'circus arts',
    'stand-up comedy', 'improv', 'theater acting',
    'opera', 'ballet', 'contemporary dance',
    'folk dance', 'ballroom dance', 'latin dance',
    'hip hop dance', 'breakdance', 'house dance',
    'locking', 'popping', 'krumping',
    'musical instruments', 'piano', 'guitar',
    'violin', 'cello', 'flute', 'clarinet',
    'saxophone', 'trumpet', 'trombone',
    'drums', 'percussion', 'synthesizer',
    'djing', 'music production', 'beatmaking',
    'songwriting', 'lyrics', 'composition',
    'arrangement', 'orchestration',
    'music theory', 'ear training', 'solfeggio',
    'vocal training', 'choir singing',
    'rap', 'hip hop', 'r&b', 'soul', 'funk',
    'disco', 'house', 'techno', 'trance',
    'drum and bass', 'dubstep', 'trap',
    'rock', 'metal', 'punk', 'grunge',
    'indie', 'alternative', 'pop', 'electropop',
    'folk', 'country', 'bluegrass', 'blues',
    'jazz', 'swing', 'bossa nova', 'samba',
    'reggae', 'ska', 'latin', 'salsa', 'bachata',
    'k-pop', 'j-pop', 'c-pop', 'mandopop',
    'bollywood', 'afrobeat', 'arabic pop',
    'classical music', 'baroque', 'romantic',
    'modern classical', 'avant-garde',
    'experimental music', 'noise music',
    'ambient', 'new age', 'meditation music',
    'soundtracks', 'video game music',
    'library music', 'stock music',
    'music licensing', 'copyright law',
    'music business', 'artist management',
    'concert promotion', 'festival organization',
    'music journalism', 'music criticism',
    'music history', 'ethnomusicology',
    'acoustics', 'psychoacoustics',
    'audio engineering', 'live sound',
    'studio recording', 'mastering engineering',
    'speaker design', 'headphone design',
    'microphone design', 'instrument making',
    'luthiery', 'piano tuning',
    'music education', 'music therapy',
    'community music', 'music technology',
    'digital audio workstations', 'plugins',
    'virtual instruments', 'samples',
    'midi', 'osc', 'max/msp', 'pure data',
    'creative coding', 'generative art',
    'algorithmic composition', 'ai music',
    'interactive installations', 'media art',
    'net art', 'digital art', 'crypto art',
    'nft art', 'virtual galleries',
    'art conservation', 'museum studies',
    'curating', 'art history', 'art criticism',
    'aesthetics', 'philosophy of art',
    'cultural studies', 'semiotics',
    'linguistics', 'sociolinguistics',
    'psycholinguistics', 'neurolinguistics',
    'applied linguistics', 'translation studies',
    'interpreting', 'localization',
    'language teaching', 'tesol', 'tefl',
    'bilingualism', 'multilingualism',
    'language acquisition', 'language disorders',
    'speech therapy', 'audiology',
    'communication sciences', 'media studies',
    'journalism', 'broadcasting', 'publishing',
    'editing', 'proofreading', 'copywriting',
    'technical writing', 'academic writing',
    'creative writing', 'fiction', 'non-fiction',
    'poetry', 'drama', 'screenwriting',
    'literary criticism', 'comparative literature',
    'world literature', 'national literatures',
    'children\'s literature', 'young adult',
    'genre fiction', 'science fiction', 'fantasy',
    'horror', 'thriller', 'mystery', 'detective',
    'romance', 'historical fiction', 'realism',
    'modernism', 'postmodernism', 'magical realism',
    'satire', 'irony', 'humor', 'tragedy',
    'epic', 'lyric', 'dramatic',
    'narrative structure', 'character development',
    'plot', 'setting', 'theme', 'symbolism',
    'metaphor', 'simile', 'imagery',
    'tone', 'voice', 'style', 'genre',
    'literary devices', 'rhetoric',
    'composition', 'grammar', 'syntax',
    'morphology', 'phonology', 'semantics',
    'pragmatics', 'discourse analysis',
    'corpus linguistics', 'computational linguistics',
    'natural language understanding',
    'machine translation', 'speech synthesis',
    'chatbots', 'virtual assistants',
    'conversational ai', 'dialogue systems',
    'information retrieval', 'search engines',
    'recommendation systems', 'personalization',
    'user profiling', 'behavioral analytics',
    'click-through rate', 'conversion rate',
    'bounce rate', 'session duration',
    'page views', 'unique visitors',
    'traffic sources', 'referral traffic',
    'organic search', 'paid search',
    'social media traffic', 'direct traffic',
    'email marketing metrics', 'open rate',
    'click rate', 'unsubscribe rate',
    'spam complaints', 'deliverability',
    'list hygiene', 'segmentation',
    'automation', 'drip campaigns',
    'lead nurturing', 'sales funnel',
    'customer journey', 'touchpoints',
    'pain points', 'value proposition',
    'unique selling proposition',
    'brand identity', 'brand voice',
    'brand positioning', 'brand equity',
    'brand loyalty', 'brand advocacy',
    'customer satisfaction', 'net promoter score',
    'customer effort score', 'customer lifetime value',
    'churn rate', 'retention rate',
    'acquisition cost', 'return on ad spend',
    'marketing mix', '4ps', '7ps',
    'swot analysis', 'pestle analysis',
    'porter\'s five forces', 'bcg matrix',
    'ansoff matrix', 'mckinsey 7s',
    'balanced scorecard', 'okrs', 'kpis',
    'agile methodology', 'scrum framework',
    'kanban board', 'lean principles',
    'six sigma belts', 'dmaic', 'dmadv',
    'kaizen', 'pdca', '5 whys', 'fishbone diagram',
    'pareto principle', 'eisenhower matrix',
    'pomodoro technique', 'time blocking',
    'getting things done', 'bullet journal',
    'habit tracking', 'goal setting techniques',
    'visualization', 'affirmations',
    'mind mapping', 'brainstorming',
    'lateral thinking', 'critical thinking',
    'problem solving', 'decision making',
    'risk assessment', 'scenario planning',
    'strategic planning', 'business planning',
    'financial planning', 'budgeting',
    'forecasting', 'variance analysis',
    'cost-benefit analysis', 'roi calculation',
    'break-even analysis', 'sensitivity analysis',
    'monte carlo simulation', 'decision trees',
    'game theory', 'behavioral economics',
    'nudges', 'choice architecture',
    'consumer psychology', 'social psychology',
    'cognitive psychology', 'developmental psychology',
    'personality psychology', 'clinical psychology',
    'counseling psychology', 'educational psychology',
    'organizational psychology', 'industrial psychology',
    'forensic psychology', 'health psychology',
    'sport psychology', 'environmental psychology',
    'positive psychology', 'humanistic psychology',
    'psychoanalysis', 'cognitive behavioral therapy',
    'dialectical behavior therapy',
    'acceptance and commitment therapy',
    'mindfulness-based stress reduction',
    'eye movement desensitization and reprocessing',
    'family therapy', 'couple therapy',
    'group therapy', 'support groups',
    'peer support', 'self-help',
    'mental health awareness', 'stigma reduction',
    'suicide prevention', 'crisis intervention',
    'trauma-informed care', 'ptsd',
    'anxiety disorders', 'depression',
    'bipolar disorder', 'schizophrenia',
    'eating disorders', 'addiction',
    'obsessive-compulsive disorder',
    'attention deficit hyperactivity disorder',
    'autism spectrum disorder',
    'learning disabilities', 'intellectual disabilities',
    'neurodiversity', 'inclusive education',
    'special education', 'early intervention',
    'child development', 'adolescent development',
    'adult development', 'aging',
    'gerontology', 'palliative care',
    'end-of-life care', 'bereavement',
    'grief counseling', 'loss',
    'resilience', 'coping strategies',
    'stress resilience', 'emotional regulation',
    'self-esteem', 'self-confidence',
    'assertiveness', 'boundaries',
    'communication skills', 'active listening',
    'empathy', 'compassion', 'kindness',
    'gratitude', 'forgiveness',
    'love', 'friendship', 'relationships',
    'dating', 'marriage', 'parenting',
    'family dynamics', 'siblings',
    'extended family', 'community',
    'social capital', 'social cohesion',
    'civic engagement', 'volunteering',
    'philanthropy', 'corporate social responsibility',
    'sustainability', 'csr', 'esg',
    'green business', 'social enterprise',
    'impact investing', 'microfinance',
    'fair trade', 'ethical sourcing',
    'supply chain transparency',
    'labor rights', 'human rights due diligence',
    'anti-corruption', 'whistleblowing',
    'compliance programs', 'ethics training',
    'code of conduct', 'values',
    'mission', 'vision', 'purpose',
    'organizational culture', 'employee engagement',
    'talent management', 'succession planning',
    'performance appraisal', 'feedback',
    'coaching', 'mentoring', 'sponsorship',
    'diversity and inclusion', 'equity',
    'belonging', 'psychological safety',
    'team dynamics', 'collaboration',
    'remote team management', 'virtual teams',
    'cross-cultural communication',
    'global leadership', 'expatriate management',
    'international business', 'trade agreements',
    'foreign direct investment',
    'multinational corporations',
    'global supply chains', 'offshoring',
    'nearshoring', 'reshoring',
    'free trade zones', 'customs unions',
    'economic integration', 'regional blocs',
    'development economics', 'poverty alleviation',
    'income inequality', 'wealth distribution',
    'social mobility', 'education access',
    'healthcare access', 'housing affordability',
    'food security', 'water security',
    'energy security', 'climate change mitigation',
    'climate change adaptation',
    'carbon footprint', 'carbon offsetting',
    'renewable energy transition',
    'circular economy', 'waste reduction',
    'sustainable agriculture', 'regenerative agriculture',
    'agroecology', 'permaculture',
    'urban farming', 'vertical farming',
    'hydroponics', 'aquaponics',
    'food waste', 'composting',
    'plastic pollution', 'ocean cleanup',
    'air quality', 'water quality',
    'soil health', 'biodiversity loss',
    'deforestation', 'desertification',
    'overfishing', 'illegal wildlife trade',
    'invasive species', 'habitat destruction',
    'pollution control', 'emissions trading',
    'environmental regulations',
    'green taxes', 'subsidies for renewables',
    'climate policy', 'paris agreement',
    'sdgs', 'sustainable development goals',
    'agenda 2030', 'global goals',
    'human development index',
    'gender equality', 'women\'s empowerment',
    'girls\' education', 'maternal health',
    'reproductive rights', 'family planning',
    'child marriage', 'female genital mutilation',
    'gender-based violence',
    'lgbtq+ rights', 'transgender rights',
    'racial justice', 'indigenous rights',
    'disability rights', 'ageism',
    'religious freedom', 'freedom of expression',
    'freedom of assembly', 'freedom of association',
    'right to privacy', 'data protection',
    'digital rights', 'internet freedom',
    'net neutrality', 'open source',
    'knowledge sharing', 'open access',
    'creative commons', 'public domain',
    'intellectual property rights',
    'patents', 'trademarks', 'copyrights',
    'trade secrets', 'licensing',
    'franchising', 'joint ventures',
    'strategic alliances', 'partnerships',
    'networking', 'relationship building',
    'trust', 'reputation', 'credibility',
    'authority', 'influence', 'persuasion',
    'negotiation', 'mediation', 'arbitration',
    'litigation', 'dispute resolution',
    'contract law', 'tort law',
    'criminal law', 'civil law',
    'constitutional law', 'administrative law',
    'international law', 'human rights law',
    'environmental law', 'labor law',
    'tax law', 'corporate law',
    'bankruptcy law', 'intellectual property law',
    'family law', 'immigration law',
    'health law', 'education law',
    'media law', 'cyber law',
    'space law', 'maritime law',
    'aviation law', 'transport law',
    'energy law', 'mining law',
    'agricultural law', 'food law',
    'consumer protection law',
    'competition law', 'antitrust law',
    'securities law', 'banking law',
    'insurance law', 'real estate law',
    'construction law', 'environmental impact assessment',
    'zoning laws', 'building codes',
    'safety regulations', 'health regulations',
    'occupational safety', 'workplace safety',
    'product safety', 'food safety standards',
    'pharmaceutical regulations',
    'medical device regulations',
    'clinical trial regulations',
    'data privacy regulations',
    'gdpr compliance', 'hipaa compliance',
    'pci dss', 'iso 27001',
    'soc 2', 'nist framework',
    'cybersecurity frameworks',
    'incident response', 'disaster recovery',
    'business continuity', 'crisis management plans',
    'emergency preparedness', 'evacuation plans',
    'first aid kits', 'emergency supplies',
    'survival skills', 'bushcraft',
    'camping', 'hiking gear',
    'outdoor recreation', 'adventure sports',
    'extreme sports', 'adrenaline junkies',
    'travel insurance', 'health insurance',
    'life insurance', 'car insurance',
    'home insurance', 'pet insurance',
    'travel visas', 'passport renewal',
    'flight booking', 'hotel booking',
    'car rental', 'train tickets',
    'bus tickets', 'cruise bookings',
    'tour packages', 'guided tours',
    'self-guided tours', 'backpacking',
    'luxury travel', 'budget travel',
    'solo travel', 'family travel',
    'business travel', 'bleisure',
    'digital detox', 'wellness retreats',
    'spa treatments', 'massages',
    'yoga retreats', 'meditation retreats',
    'silent retreats', 'spiritual journeys',
    'pilgrimages', 'religious tourism',
    'cultural tourism', 'heritage sites',
    'museums', 'galleries', 'theaters',
    'concert halls', 'stadiums',
    'amusement parks', 'zoos', 'aquariums',
    'botanical gardens', 'national parks',
    'nature reserves', 'wildlife sanctuaries',
    'eco-lodges', 'glamping',
    'homestays', 'vacation rentals',
    'hostels', 'guesthouses',
    'bed and breakfast', 'boutique hotels',
    'resorts', 'all-inclusive',
    'city breaks', 'weekend getaways',
    'staycations', 'road trips',
    'camping trips', 'fishing trips',
    'hunting trips', 'safaris',
    'expeditions', 'trekking',
    'mountaineering expeditions',
    'polar expeditions', 'deep sea diving',
    'skydiving', 'bungee jumping',
    'paragliding', 'hang gliding',
    'base jumping', 'wing suit flying',
    'rock climbing competitions',
    'surfing competitions',
    'skateboarding competitions',
    'snowboarding competitions',
    'skiing competitions',
    'figure skating competitions',
    'speed skating competitions',
    'hockey tournaments',
    'football world cup',
    'olympics', 'paralympics',
    'commonwealth games',
    'asian games', 'pan american games',
    'european championships',
    'world championships',
    'grand slam tournaments',
    'formula 1', 'motogp', 'nascar',
    'rally racing', 'endurance racing',
    'karting', 'motocross',
    'superbike racing', 'touring car racing',
    'drag racing', 'drift racing',
    'off-road racing', 'truck racing',
    'boat racing', 'jet ski racing',
    'horse racing', 'greyhound racing',
    'camel racing', 'dog sledding',
    'e-sports', 'competitive gaming',
    'twitch streaming', 'youtube gaming',
    'game streaming', 'live streaming',
    'content creation', 'influencers',
    'social media platforms',
    'facebook', 'instagram', 'twitter',
    'tiktok', 'linkedin', 'snapchat',
    'pinterest', 'reddit', 'discord',
    'telegram', 'whatsapp', 'signal',
    'wechat', 'line', 'kakao talk',
    'viber', 'skype', 'zoom',
    'microsoft teams', 'slack',
    'google meet', 'webex',
    'video conferencing', 'remote collaboration',
    'cloud storage', 'file sharing',
    'document collaboration',
    'project management tools',
    'task management', 'time tracking',
    'note taking', 'knowledge management',
    'personal wiki', 'second brain',
    'productivity apps', 'habit trackers',
    'finance apps', 'budgeting apps',
    'investment apps', 'crypto wallets',
    'shopping apps', 'delivery apps',
    'ride-hailing apps', 'food delivery',
    'grocery delivery', 'pharmacy delivery',
    'travel apps', 'booking apps',
    'language learning apps',
    'fitness apps', 'meditation apps',
    'sleep apps', 'health tracking',
    'period trackers', 'pregnancy apps',
    'baby tracking', 'parenting apps',
    'education apps', 'e-learning platforms',
    'moocs', 'online courses',
    'certifications', 'degrees online',
    'virtual classrooms', 'webinars',
    'online workshops', 'masterclasses',
    'tutoring', 'coaching online',
    'consulting', 'freelancing platforms',
    'job boards', 'career sites',
    'professional networks',
    'industry associations',
    'trade unions', 'guilds',
    'chambers of commerce',
    'business clubs', 'rotary club',
    'lion\'s club', 'kiwanis',
    'service organizations',
    'charities', 'non-profits',
    'foundations', 'trusts',
    'endowments', 'grants',
    'fundraising', 'crowdfunding',
    'donations', 'volunteering opportunities',
    'community service',
    'local events', 'festivals',
    'concerts', 'shows',
    'exhibitions', 'fairs',
    'markets', 'bazaars',
    'conferences', 'summits',
    'meetups', 'networking events',
    'hackathons', 'competitions',
    'contests', 'challenges',
    'awards', 'prizes',
    'scholarships', 'fellowships',
    'internships', 'apprenticeships',
    'traineeships', 'graduate programs',
    'entry-level jobs', 'junior roles',
    'mid-level jobs', 'senior roles',
    'executive roles', 'c-suite',
    'board positions', 'advisory roles',
    'consulting roles', 'freelance gigs',
    'part-time jobs', 'full-time jobs',
    'remote jobs', 'hybrid jobs',
    'on-site jobs', 'shift work',
    'seasonal work', 'temporary work',
    'contract work', 'project-based work',
    'gig work', 'platform work',
    'self-employment', 'entrepreneurship',
    'small business', 'startups',
    'scale-ups', 'unicorns',
    'ipo', 'acquisition', 'merger',
    'bankruptcy', 'liquidation',
    'restructuring', 'turnaround',
    'growth strategy', 'market entry',
    'product launch', 'brand launch',
    'campaign management',
    'advertising agencies',
    'pr agencies', 'marketing agencies',
    'digital agencies', 'creative agencies',
    'media buying', 'media planning',
    'audience targeting', 'retargeting',
    'lookalike audiences',
    'conversion optimization',
    'landing page optimization',
    'a/b testing', 'multivariate testing',
    'user experience testing',
    'usability testing', 'accessibility testing',
    'performance testing', 'load testing',
    'security testing', 'penetration testing',
    'code review', 'peer review',
    'quality assurance', 'test automation',
    'continuous integration',
    'continuous deployment',
    'devops culture', 'agile culture',
    'innovation culture', 'learning culture',
    'feedback culture', 'recognition culture',
    'well-being culture', 'diversity culture',
    'sustainability culture',
    'ethical culture', 'compliance culture',
    'safety culture', 'quality culture',
    'customer-centric culture',
    'data-driven culture',
    'digital-first culture',
    'remote-first culture',
    'hybrid-work culture',
    'flexible-work culture',
    'results-only work environment',
    'four-day work week',
    'unlimited vacation',
    'sabbaticals', 'leave policies',
    'parental leave', 'maternity leave',
    'paternity leave', 'adoption leave',
    'caregiver leave', 'bereavement leave',
    'sick leave', 'mental health days',
    'volunteer time off',
    'professional development budget',
    'tuition reimbursement',
    'conference attendance',
    'certification support',
    'mentorship programs',
    'leadership development',
    'succession planning',
    'talent pipeline',
    'employer branding',
    'candidate experience',
    'recruitment marketing',
    'talent acquisition',
    'headhunting', 'executive search',
    'staffing agencies',
    'recruitment process outsourcing',
    'applicant tracking systems',
    'hr information systems',
    'payroll systems',
    'benefits administration',
    'performance management systems',
    'learning management systems',
    'employee engagement platforms',
    'internal communication tools',
    'intranets', 'employee portals',
    'knowledge bases', 'wikis',
    'document management systems',
    'content management systems',
    'digital asset management',
    'customer relationship management',
    'sales force automation',
    'marketing automation',
    'service desk software',
    'help desk software',
    'ticketing systems',
    'live chat software',
    'chatbot platforms',
    'voice assistants',
    'smart speakers',
    'wearable technology',
    'internet of things devices',
    'smart home devices',
    'connected cars',
    'industrial iot sensors',
    'smart city infrastructure',
    'smart grid technology',
    'smart metering',
    'smart lighting',
    'smart parking',
    'smart waste management',
    'smart water management',
    'smart agriculture',
    'smart healthcare',
    'smart education',
    'smart retail',
    'smart logistics',
    'smart manufacturing',
    'industry 4.0 technologies',
    'digital twins in manufacturing',
    'predictive maintenance in industry',
    'robotics in manufacturing',
    'cobots', 'automated guided vehicles',
    '3d printing in manufacturing',
    'additive manufacturing',
    'subtractive manufacturing',
    'cnc machining',
    'laser cutting', 'water jet cutting',
    'injection molding', 'die casting',
    'extrusion', 'blow molding',
    'thermoforming', 'vacuum forming',
    'composite materials',
    'carbon fiber', 'fiberglass',
    'kevlar', 'graphene',
    'nanomaterials', 'biomaterials',
    'smart materials', 'shape memory alloys',
    'piezoelectric materials',
    'thermoelectric materials',
    'photovoltaic materials',
    'battery materials',
    'fuel cell materials',
    'hydrogen storage materials',
    'catalysts', 'enzymes',
    'polymers', 'plastics',
    'rubber', 'elastomers',
    'ceramics', 'glass',
    'metals', 'alloys',
    'steel', 'aluminum', 'copper',
    'titanium', 'magnesium',
    'precious metals', 'rare earth elements',
    'mining', 'extraction',
    'refining', 'smelting',
    'metallurgy', 'materials science',
    'chemical engineering',
    'process engineering',
    'plant design',
    'pipeline design',
    'pump selection',
    'valve selection',
    'instrumentation',
    'control systems',
    'plc', 'scada',
    'dcs', 'sis',
    'hmi', 'operator interface',
    'alarm management',
    'safety instrumented systems',
    'hazard analysis',
    'hazop', 'loh',
    'layer of protection analysis',
    'quantitative risk assessment',
    'process safety management',
    'occupational health and safety',
    'environmental health and safety',
    'iso 45001', 'iso 14001',
    'iso 9001', 'quality management systems',
    'total quality management',
    'continuous improvement',
    'operational excellence',
    'lean six sigma',
    'theory of constraints',
    'critical chain project management',
    'earned value management',
    'project portfolio management',
    'program management',
    'change management',
    'stakeholder management',
    'communication management',
    'risk management',
    'procurement management',
    'contract management',
    'scope management',
    'schedule management',
    'cost management',
    'resource management',
    'integration management',
    'pmbok', 'prince2',
    'agile project management',
    'scrumban', 'xp',
    'feature driven development',
    'dynamic systems development method',
    'crystal methods',
    'adaptive software development',
    'rapid application development',
    'joint application development',
    'structured systems analysis and design method',
    'object-oriented analysis and design',
    'unified modeling language',
    'business process modeling notation',
    'system dynamics',
    'soft systems methodology',
    'rich picture',
    'root cause analysis',
    'force field analysis',
    'stakeholder analysis',
    'swot analysis',
    'pestle analysis',
    'porter\'s five forces',
    'value chain analysis',
    'core competence analysis',
    'resource-based view',
    'dynamic capabilities',
    'blue ocean strategy',
    'red ocean strategy',
    'disruptive innovation',
    'sustaining innovation',
    'incremental innovation',
    'radical innovation',
    'architectural innovation',
    'modular innovation',
    'open innovation',
    'closed innovation',
    'user innovation',
    'democratizing innovation',
    'frugal innovation',
    'reverse innovation',
    'social innovation',
    'business model innovation',
    'platform business models',
    'ecosystem strategy',
    'network effects',
    'two-sided markets',
    'multi-sided platforms',
    'sharing economy platforms',
    'gig economy platforms',
    'creator economy platforms',
    'attention economy',
    'data economy',
    'algorithmic economy',
    'token economy',
    'cryptoeconomics',
    'mechanism design',
    'game theory applications',
    'auction theory',
    'market design',
    'matching markets',
    'two-sided matching',
    'stable marriage problem',
    'college admissions',
    'kidney exchange',
    'school choice',
    'labor market matching',
    'housing market matching',
    'organ donation',
    'blood donation',
    'volunteer matching',
    'mentor matching',
    'investor-startup matching',
    'job-candidate matching',
    'supplier-buyer matching',
    'landlord-tenant matching',
    'driver-rider matching',
    'host-guest matching',
    'seller-buyer matching',
    'advertiser-publisher matching',
    'content creator-viewer matching',
    'teacher-student matching',
    'doctor-patient matching',
    'lawyer-client matching',
    'consultant-company matching',
    'freelancer-client matching',
    'expert-seeker matching',
    'problem-solver matching',
    'idea-generator matching',
    'funder-project matching',
    'donor-cause matching',
    'volunteer-opportunity matching',
    'event-attendee matching',
    'conference-speaker matching',
    'panelist-moderator matching',
    'interviewer-interviewee matching',
    'reviewer-author matching',
    'editor-writer matching',
    'publisher-book matching',
    'agent-talent matching',
    'producer-artist matching',
    'director-actor matching',
    'coach-athlete matching',
    'trainer-trainee matching',
    'guide-tourist matching',
    'chef-diner matching',
    'stylist-client matching',
    'designer-homeowner matching',
    'architect-builder matching',
    'contractor-subcontractor matching',
    'supplier-manufacturer matching',
    'distributor-retailer matching',
    'wholesaler-retailer matching',
    'importer-exporter matching',
    'broker-client matching',
    'agent-principal matching',
    'franchisor-franchisee matching',
    'licensor-licensee matching',
    'partner-partner matching',
    'joint-venture-partner matching',
    'strategic-alliance-partner matching',
    'merger-acquisition-target matching',
    'investor-company matching',
    'lender-borrower matching',
    'insurer-insured matching',
    'reinsurer-insurer matching',
    'bank-customer matching',
    'credit-card-issuer-cardholder matching',
    'payment-processor-merchant matching',
    'exchange-trader matching',
    'clearinghouse-member matching',
    'custodian-investor matching',
    'prime-broker-hedge-fund matching',
    'market-maker-trader matching',
    'liquidity-provider-trader matching',
    'order-book-matching',
    'continuous-double-auction',
    'call-auction',
    'uniform-price-auction',
    'discriminatory-price-auction',
    'vikrey-auction',
    'english-auction',
    'dutch-auction',
    'sealed-bid-auction',
    'procurement-auction',
    'spectrum-auction',
    'ad-auction',
    'keyword-auction',
    'display-ad-auction',
    'video-ad-auction',
    'native-ad-auction',
    'social-ad-auction',
    'search-ad-auction',
    'mobile-ad-auction',
    'programmatic-advertising',
    'real-time-bidding',
    'private-marketplace',
    'preferred-deal',
    'programmatic-direct',
    'header-bidding',
    'ad-exchange',
    'demand-side-platform',
    'supply-side-platform',
    'data-management-platform',
    'customer-data-platform',
    'identity-resolution',
    'attribution-modeling',
    'marketing-mix-modeling',
    'lift-studies',
    'brand-lift-studies',
    'conversion-lift-studies',
    'sales-lift-studies',
    'awareness-lift-studies',
    'consideration-lift-studies',
    'intent-lift-studies',
    'loyalty-lift-studies',
    'advocacy-lift-studies',
    'retention-lift-studies',
    'churn-lift-studies',
    'engagement-lift-studies',
    'reach-lift-studies',
    'frequency-lift-studies',
    'impression-lift-studies',
    'click-lift-studies',
    'view-through-lift-studies',
    'cross-device-lift-studies',
    'cross-channel-lift-studies',
    'omnichannel-lift-studies',
    'offline-lift-studies',
    'online-lift-studies',
    'digital-lift-studies',
    'traditional-media-lift-studies',
    'tv-lift-studies',
    'radio-lift-studies',
    'print-lift-studies',
    'out-of-home-lift-studies',
    'cinema-lift-studies',
    'event-lift-studies',
    'experiential-lift-studies',
    'influencer-lift-studies',
    'content-lift-studies',
    'seo-lift-studies',
    'sem-lift-studies',
    'social-media-lift-studies',
    'email-lift-studies',
    'sms-lift-studies',
    'push-notification-lift-studies',
    'in-app-lift-studies',
    'website-lift-studies',
    'app-lift-studies',
    'landing-page-lift-studies',
    'checkout-lift-studies',
    'cart-abandonment-lift-studies',
    'product-page-lift-studies',
    'category-page-lift-studies',
    'homepage-lift-studies',
    'blog-lift-studies',
    'forum-lift-studies',
    'community-lift-studies',
    'support-lift-studies',
    'documentation-lift-studies',
    'tutorial-lift-studies',
    'webinar-lift-studies',
    'podcast-lift-studies',
    'video-lift-studies',
    'image-lift-studies',
    'infographic-lift-studies',
    'ebook-lift-studies',
    'whitepaper-lift-studies',
    'case-study-lift-studies',
    'testimonial-lift-studies',
    'review-lift-studies',
    'rating-lift-studies',
    'survey-lift-studies',
    'poll-lift-studies',
    'quiz-lift-studies',
    'contest-lift-studies',
    'giveaway-lift-studies',
    'sweepstakes-lift-studies',
    'lottery-lift-studies',
    'raffle-lift-studies',
    'auction-lift-studies',
    'bid-lift-studies',
    'offer-lift-studies',
    'discount-lift-studies',
    'coupon-lift-studies',
    'promo-code-lift-studies',
    'deal-lift-studies',
    'bundle-lift-studies',
    'upsell-lift-studies',
    'cross-sell-lift-studies',
    'downsell-lift-studies',
    'subscription-lift-studies',
    'membership-lift-studies',
    'loyalty-program-lift-studies',
    'referral-program-lift-studies',
    'affiliate-program-lift-studies',
    'partner-program-lift-studies',
    'reseller-program-lift-studies',
    'distributor-program-lift-studies',
    'franchise-program-lift-studies',
    'licensing-program-lift-studies',
    'joint-venture-program-lift-studies',
    'strategic-alliance-program-lift-studies',
    'merger-acquisition-program-lift-studies',
    'investment-program-lift-studies',
    'financing-program-lift-studies',
    'insurance-program-lift-studies',
    'banking-program-lift-studies',
    'payment-program-lift-studies',
    'exchange-program-lift-studies',
    'clearing-program-lift-studies',
    'custody-program-lift-studies',
    'prime-brokerage-program-lift-studies',
    'market-making-program-lift-studies',
    'liquidity-provision-program-lift-studies',
    'order-routing-program-lift-studies',
    'execution-program-lift-studies',
    'settlement-program-lift-studies',
    'reporting-program-lift-studies',
    'compliance-program-lift-studies',
    'risk-management-program-lift-studies',
    'audit-program-lift-studies',
    'tax-program-lift-studies',
    'legal-program-lift-studies',
    'hr-program-lift-studies',
    'it-program-lift-studies',
    'security-program-lift-studies',
    'privacy-program-lift-studies',
    'data-governance-program-lift-studies',
    'quality-program-lift-studies',
    'safety-program-lift-studies',
    'environmental-program-lift-studies',
    'social-program-lift-studies',
    'governance-program-lift-studies',
    'esg-program-lift-studies',
    'sustainability-program-lift-studies',
    'csr-program-lift-studies',
    'philanthropy-program-lift-studies',
    'volunteering-program-lift-studies',
    'community-engagement-program-lift-studies',
    'stakeholder-engagement-program-lift-studies',
    'customer-engagement-program-lift-studies',
    'employee-engagement-program-lift-studies',
    'investor-engagement-program-lift-studies',
    'partner-engagement-program-lift-studies',
    'supplier-engagement-program-lift-studies',
    'regulator-engagement-program-lift-studies',
    'media-engagement-program-lift-studies',
    'analyst-engagement-program-lift-studies',
    'industry-engagement-program-lift-studies',
    'academic-engagement-program-lift-studies',
    'ngo-engagement-program-lift-studies',
    'government-engagement-program-lift-studies',
    'public-engagement-program-lift-studies',
    'global-engagement-program-lift-studies',
    'local-engagement-program-lift-studies',
    'regional-engagement-program-lift-studies',
    'national-engagement-program-lift-studies',
    'international-engagement-program-lift-studies',
    'cross-border-engagement-program-lift-studies',
    'multi-cultural-engagement-program-lift-studies',
    'multi-lingual-engagement-program-lift-studies',
    'multi-channel-engagement-program-lift-studies',
    'omni-channel-engagement-program-lift-studies',
    'digital-engagement-program-lift-studies',
    'physical-engagement-program-lift-studies',
    'hybrid-engagement-program-lift-studies',
    'virtual-engagement-program-lift-studies',
    'augmented-reality-engagement-program-lift-studies',
    'virtual-reality-engagement-program-lift-studies',
    'mixed-reality-engagement-program-lift-studies',
    'extended-reality-engagement-program-lift-studies',
    'metaverse-engagement-program-lift-studies',
    'web3-engagement-program-lift-studies',
    'blockchain-engagement-program-lift-studies',
    'crypto-engagement-program-lift-studies',
    'nft-engagement-program-lift-studies',
    'dao-engagement-program-lift-studies',
    'defi-engagement-program-lift-studies',
    'gamefi-engagement-program-lift-studies',
    'socialfi-engagement-program-lift-studies',
    'creator-economy-engagement-program-lift-studies',
    'influencer-economy-engagement-program-lift-studies',
    'gig-economy-engagement-program-lift-studies',
    'sharing-economy-engagement-program-lift-studies',
    'subscription-economy-engagement-program-lift-studies',
    'experience-economy-engagement-program-lift-studies',
    'attention-economy-engagement-program-lift-studies',
    'data-economy-engagement-program-lift-studies',
    'algorithmic-economy-engagement-program-lift-studies',
    'token-economy-engagement-program-lift-studies',
    'cryptoeconomy-engagement-program-lift-studies',
    'mechanism-design-engagement-program-lift-studies',
    'game-theory-engagement-program-lift-studies',
    'auction-theory-engagement-program-lift-studies',
    'market-design-engagement-program-lift-studies',
    'matching-markets-engagement-program-lift-studies',
    'two-sided-matching-engagement-program-lift-studies',
    'stable-marriage-problem-engagement-program-lift-studies',
    'college-admissions-engagement-program-lift-studies',
    'kidney-exchange-engagement-program-lift-studies',
    'school-choice-engagement-program-lift-studies',
    'labor-market-matching-engagement-program-lift-studies',
    'housing-market-matching-engagement-program-lift-studies',
    'organ-donation-engagement-program-lift-studies',
    'blood-donation-engagement-program-lift-studies',
    'volunteer-matching-engagement-program-lift-studies',
    'mentor-matching-engagement-program-lift-studies',
    'investor-startup-matching-engagement-program-lift-studies',
    'job-candidate-matching-engagement-program-lift-studies',
    'supplier-buyer-matching-engagement-program-lift-studies',
    'landlord-tenant-matching-engagement-program-lift-studies',
    'driver-rider-matching-engagement-program-lift-studies',
    'host-guest-matching-engagement-program-lift-studies',
    'seller-buyer-matching-engagement-program-lift-studies',
    'advertiser-publisher-matching-engagement-program-lift-studies',
    'content-creator-viewer-matching-engagement-program-lift-studies',
    'teacher-student-matching-engagement-program-lift-studies',
    'doctor-patient-matching-engagement-program-lift-studies',
    'lawyer-client-matching-engagement-program-lift-studies',
    'consultant-company-matching-engagement-program-lift-studies',
    'freelancer-client-matching-engagement-program-lift-studies',
    'expert-seeker-matching-engagement-program-lift-studies',
    'problem-solver-matching-engagement-program-lift-studies',
    'idea-generator-matching-engagement-program-lift-studies',
    'funder-project-matching-engagement-program-lift-studies',
    'donor-cause-matching-engagement-program-lift-studies',
    'volunteer-opportunity-matching-engagement-program-lift-studies',
    'event-attendee-matching-engagement-program-lift-studies',
    'conference-speaker-matching-engagement-program-lift-studies',
    'panelist-moderator-matching-engagement-program-lift-studies',
    'interviewer-interviewee-matching-engagement-program-lift-studies',
    'reviewer-author-matching-engagement-program-lift-studies',
    'editor-writer-matching-engagement-program-lift-studies',
    'publisher-book-matching-engagement-program-lift-studies',
    'agent-talent-matching-engagement-program-lift-studies',
    'producer-artist-matching-engagement-program-lift-studies',
    'director-actor-matching-engagement-program-lift-studies',
    'coach-athlete-matching-engagement-program-lift-studies',
    'trainer-trainee-matching-engagement-program-lift-studies',
    'guide-tourist-matching-engagement-program-lift-studies',
    'chef-diner-matching-engagement-program-lift-studies',
    'stylist-client-matching-engagement-program-lift-studies',
    'designer-homeowner-matching-engagement-program-lift-studies',
    'architect-builder-matching-engagement-program-lift-studies',
    'contractor-subcontractor-matching-engagement-program-lift-studies',
    'supplier-manufacturer-matching-engagement-program-lift-studies',
    'distributor-retailer-matching-engagement-program-lift-studies',
    'wholesaler-retailer-matching-engagement-program-lift-studies',
    'importer-exporter-matching-engagement-program-lift-studies',
    'broker-client-matching-engagement-program-lift-studies',
    'agent-principal-matching-engagement-program-lift-studies',
    'franchisor-franchisee-matching-engagement-program-lift-studies',
    'licensor-licensee-matching-engagement-program-lift-studies',
    'partner-partner-matching-engagement-program-lift-studies',
    'joint-venture-partner-matching-engagement-program-lift-studies',
    'strategic-alliance-partner-matching-engagement-program-lift-studies',
    'merger-acquisition-target-matching-engagement-program-lift-studies',
    'investor-company-matching-engagement-program-lift-studies',
    'lender-borrower-matching-engagement-program-lift-studies',
    'insurer-insured-matching-engagement-program-lift-studies',
    'reinsurer-insurer-matching-engagement-program-lift-studies',
    'bank-customer-matching-engagement-program-lift-studies',
    'credit-card-issuer-cardholder-matching-engagement-program-lift-studies',
    'payment-processor-merchant-matching-engagement-program-lift-studies',
    'exchange-trader-matching-engagement-program-lift-studies',
    'clearinghouse-member-matching-engagement-program-lift-studies',
    'custodian-investor-matching-engagement-program-lift-studies',
    'prime-broker-hedge-fund-matching-engagement-program-lift-studies',
    'market-maker-trader-matching-engagement-program-lift-studies',
    'liquidity-provider-trader-matching-engagement-program-lift-studies',
    'order-book-matching-engagement-program-lift-studies',
    'continuous-double-auction-engagement-program-lift-studies',
    'call-auction-engagement-program-lift-studies',
    'uniform-price-auction-engagement-program-lift-studies',
    'discriminatory-price-auction-engagement-program-lift-studies',
    'vikrey-auction-engagement-program-lift-studies',
    'english-auction-engagement-program-lift-studies',
    'dutch-auction-engagement-program-lift-studies',
    'sealed-bid-auction-engagement-program-lift-studies',
    'procurement-auction-engagement-program-lift-studies',
    'spectrum-auction-engagement-program-lift-studies',
    'ad-auction-engagement-program-lift-studies',
    'keyword-auction-engagement-program-lift-studies',
    'display-ad-auction-engagement-program-lift-studies',
    'video-ad-auction-engagement-program-lift-studies',
    'native-ad-auction-engagement-program-lift-studies',
    'social-ad-auction-engagement-program-lift-studies',
    'search-ad-auction-engagement-program-lift-studies',
    'mobile-ad-auction-engagement-program-lift-studies',
    'programmatic-advertising-engagement-program-lift-studies',
    'real-time-bidding-engagement-program-lift-studies',
    'private-marketplace-engagement-program-lift-studies',
    'preferred-deal-engagement-program-lift-studies',
    'programmatic-direct-engagement-program-lift-studies',
    'header-bidding-engagement-program-lift-studies',
    'ad-exchange-engagement-program-lift-studies',
    'demand-side-platform-engagement-program-lift-studies',
    'supply-side-platform-engagement-program-lift-studies',
    'data-management-platform-engagement-program-lift-studies',
    'customer-data-platform-engagement-program-lift-studies',
    'identity-resolution-engagement-program-lift-studies',
    'attribution-modeling-engagement-program-lift-studies',
    'marketing-mix-modeling-engagement-program-lift-studies',
    'lift-studies-engagement-program-lift-studies',
}

# функция извлечения заголовка
def extract_title_heuristic(query):
    if not isinstance(query, str):
        return np.nan
    
    words = query.split()
    # Фильтруем слова, которые не являются стоп-словами поиска
    filtered_words = [w for w in words if w not in stop_words_search and len(w) > 1]
    
    if not filtered_words:
        return np.nan
        
    title = " ".join(filtered_words)
    
    if len(title) < 2:
        return np.nan
        
    return title

with open(os.path.join(MODEL_DIR, 'stop_words_search.pkl'), 'wb') as f:
    pickle.dump(stop_words_search, f)

In [2]:
%%writefile solution.py
import os
import re
import pickle
import numpy as np
import pandas as pd
import gc

class PredictionModel:
    batch_size: int = 1024
    
    def __init__(self) -> None:
        self.model_dir = '/kaggle/working/models'
        
        with open(os.path.join(self.model_dir, 'tfidf_type.pkl'), 'rb') as f:
            self.tfidf_type = pickle.load(f)
        with open(os.path.join(self.model_dir, 'model_type.pkl'), 'rb') as f:
            self.model_type = pickle.load(f)
        
        with open(os.path.join(self.model_dir, 'tfidf_content.pkl'), 'rb') as f:
            self.tfidf_content = pickle.load(f)
        with open(os.path.join(self.model_dir, 'model_content.pkl'), 'rb') as f:
            self.model_content = pickle.load(f)
        with open(os.path.join(self.model_dir, 'le_content.pkl'), 'rb') as f:
            self.le_content = pickle.load(f)
        
        with open(os.path.join(self.model_dir, 'stop_words_search.pkl'), 'rb') as f:
            self.stop_words_search = pickle.load(f)

    def _clean_text(self, text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zа-яё0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _extract_title(self, query):
        if not isinstance(query, str) or pd.isna(query):
            return np.nan
        
        words = query.split()
        filtered_words = [w for w in words if w not in self.stop_words_search and len(w) > 1]
        
        if not filtered_words:
            return np.nan
            
        title = " ".join(filtered_words)
        
        if len(title) < 2:
            return np.nan
            
        return title

    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df[["QueryText"]].copy()
        
        cleaned_queries = df['QueryText'].apply(self._clean_text)
        
        X_type = self.tfidf_type.transform(cleaned_queries)
        pred_type = self.model_type.predict(X_type)
        prob_type = self.model_type.predict_proba(X_type)[:, 1]
        
        out["TypeQuery"] = pred_type
        
        out["Title"] = np.nan
        out["ContentType"] = np.nan
        
        idx_positive = out[out["TypeQuery"] == 1].index
        
        if len(idx_positive) > 0:
            positive_queries = cleaned_queries.loc[idx_positive]
            
            X_content = self.tfidf_content.transform(positive_queries)
            pred_content_enc = self.model_content.predict(X_content)
            pred_content_labels = self.le_content.inverse_transform(pred_content_enc)
            
            out.loc[idx_positive, "ContentType"] = pred_content_labels
            
            titles = positive_queries.apply(self._extract_title)
            out.loc[idx_positive, "Title"] = titles
            
        gc.collect()
        
        return out

Writing solution.py


In [3]:
import numpy as np
import pandas as pd

from solution import PredictionModel

train_path = '/kaggle/input/datasets/antonoof/traindata-media/train.csv'

try:
    df_train = pd.read_csv(train_path)
except Exception as e:
    print(f"Ошибка загрузки: {e}")

model = PredictionModel()

predictions_df = model.predict(df_train)
predictions_df.head(5)

/kaggle/working/solution.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['сериал' 'фильм' 'фильм' ... 'сериал' 'сериал' 'мультсериал']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.loc[idx_positive, "ContentType"] = pred_content_labels
/kaggle/working/solution.py:80: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['белая королева 2013' 'омен 2006 русском'
 'хищники против чужого без торрента' ... 'восемнадцать'
 'сериа территория' 'грифины']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.loc[idx_positive, "Title"] = titles


,QueryText,TypeQuery,Title,ContentType
0,белая королева сериал 2013,1,белая королева 2013,сериал
1,омен 2006 смотреть онлайн на русском,1,омен 2006 русском,фильм
2,анонимный чат с фото,0,NaN,NaN
3,хищники против чужого скачать без торрента,1,хищники против чужого без торрента,фильм
4,скачать гравити фолз на 2 часа,1,гравити фолз часа,мультсериал


In [4]:
df_train.head(5)

,QueryText,TypeQuery,Title,ContentType
0,белая королева сериал 2013,1,белая королева,сериал
1,омен 2006 смотреть онлайн на русском,1,омен,фильм
2,анонимный чат с фото,0,NaN,NaN
3,хищники против чужого скачать без торрента,1,чужие против хищника,фильм
4,скачать гравити фолз на 2 часа,1,гравити фолз,мультсериал


In [5]:
import zipfile
import os

MODEL_DIR = '/kaggle/working/models'
zip_path = '/kaggle/working/models.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(MODEL_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(MODEL_DIR))
            zipf.write(file_path, arcname)